# Audit label dengan lirik: I Surrender

Buka dari root **SongMap** lalu Run All. Contoh input sudah disertakan dari JSON Anda, transkripsi MP3 yang telah saya jalankan secara lokal, dan rujukan section/lirik yang Anda berikan. **Tidak perlu unggah audio, ekspor JSON, menjalankan ASR atau mengisi anotasi lagi untuk contoh ini.**

Audit mempertahankan boundary Apple dan label model. Frasa lirik yang cocok digunakan sebagai bukti review untuk pengulangan bridge, akhir chorus di dalam inst, atau rentang yang mencampur beberapa fungsi. Skor kemiripan teks bukan confidence klasifikasi. Referensi lirik bisa berbeda dengan pertunjukan; urutan dan jumlah pengulangannya tidak dipaksakan.

Contoh ini memakai transkripsi otomatis yang memiliki salah kata dan timestamp perkiraan. Onset vokal bisa mendahului garis bar. Hasil audit tidak mengubah model atau section di iPhone, dan bukan ukuran akurasi dataset.


Audit contoh ini **sudah saya jalankan**. Hasil tersimpan di `experiments/structure/lyric_review_examples/i-surrender-review.json` dan terikat hash input.

Temuan review:

- **02:35.76–02:55.22:** frasa chorus masuk ke blok inst; bagian berikutnya juga memuat awal frasa bridge.
- **03:34.14–04:13.06:** frasa bridge yang berulang bertentangan dengan label chorus.
- **04:32.52–05:14.60:** frasa bridge kembali bertentangan dengan label chorus, dengan coverage berbeda di tiap potongan.
- **00:19.55–00:58.46:** dua referensi verse cocok dalam satu blok verse; perlu menentukan granularitas, tanpa menganggap Apple otomatis salah.

Ini daftar kandidat review, bukan koreksi label/boundary yang telah diterapkan. Anda dapat membaca hasil yang sudah disimpan atau Run All untuk mengulang audit secara lokal.


In [ ]:
from pathlib import Path
from datetime import datetime
import json, subprocess, sys

ROOT=Path.cwd().resolve()
SCRIPT=ROOT/'scripts/review_lyric_structure.py'
INPUT=ROOT/'experiments/structure/lyric_review_examples/i-surrender.json'
assert SCRIPT.is_file() and INPUT.is_file(), 'Buka dari root SongMap setelah pull branch worship-hybrid-integration'
OUTPUT=ROOT/'outputs.local'/('lyric-review-'+datetime.now().strftime('%Y%m%d-%H%M%S-%f'))
command=[sys.executable,'-u',str(SCRIPT),'--input',str(INPUT),'--output-dir',str(OUTPUT)]
log=Path(str(OUTPUT)+'.log');log.parent.mkdir(parents=True,exist_ok=True)
with log.open('w',encoding='utf-8') as handle:
    process=subprocess.Popen(command,cwd=ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in process.stdout:
            print(line,end='');handle.write(line);handle.flush()
        status=process.wait()
    except KeyboardInterrupt:
        process.terminate();process.wait();raise
if status:
    raise RuntimeError(f'Gagal ({status}); lihat {log}')
print('Hasil:',OUTPUT)


In [ ]:
from IPython.display import display, Markdown
import html

review=json.loads((OUTPUT/'review.json').read_text())
def clock(s):return f'{int(s//60):02d}:{s%60:05.2f}'
notes={
 'no_usable_lyric_evidence':'Bukti lirik belum cukup',
 'compatible_with_matched_lyrics':'Sesuai frasa yang cocok',
 'same_role_multiple_references_review_granularity':'Pembagian antar-verse perlu ditentukan',
 'mixed_lyric_roles_requires_review':'Beberapa fungsi tercampur',
 'label_disagreement_requires_review':'Kandidat salah label',
 'partial_lyric_conflict_requires_review':'Sebagian rentang bertentangan',
}
lines=['| Waktu | Label app | Fungsi pada frasa cocok | Review |','|---|---|---|---|']
for row in review['sections']:
    if row['status'] in ('compatible_with_matched_lyrics','no_usable_lyric_evidence'):
        continue
    roles=', '.join(f'{label} {fraction:.0%}' for label,fraction in row['matched_role_coverage'].items())
    lines.append(f"| {clock(row['start'])}–{clock(row['end'])} | {row['label']} | {roles} | {notes[row['status']]} |")
display(Markdown('\n'.join(lines)))
print('Persentase adalah cakupan frasa yang cocok di dalam section, bukan confidence label.')
print('Review:',OUTPUT/'review.json')
print('CSV:',OUTPUT/'section_review.csv')


In [ ]:
# Bukti teks untuk setiap rentang yang perlu review.
for row in review['sections']:
    if row['status'] in ('compatible_with_matched_lyrics','no_usable_lyric_evidence'):
        continue
    print(f"\n{clock(row['start'])}–{clock(row['end'])} | app={row['label']} | {notes[row['status']]}")
    for hit in row['matches']:
        print(f"  {clock(hit['start'])}–{clock(hit['end'])}: reference={hit['label']}; similarity={hit['text_similarity']:.2f}")
        print('   Rujukan:',hit['reference_phrase'])
        print('   ASR:',hit['asr_text'])


Interpretasi:

- `inst` dengan hanya sebagian frasa chorus tidak otomatis diganti menjadi chorus untuk seluruh rentang.
- Frasa bridge yang berulang menjadi kandidat review saat model memberi label chorus.
- Verse 1 dan Verse 2 yang cocok di satu blok verse adalah persoalan granularitas. Itu tidak otomatis membuktikan boundary Apple salah.
- Frasa pendek seperti “I surrender”, frasa yang dipakai untuk beberapa kelas, serta kata ASR yang terlalu lemah tidak dipakai sebagai penentu.
- Ketiadaan kecocokan tidak membuktikan section instrumental/outro. Ad-lib dan transkripsi yang hilang perlu diperiksa.

Tahap berikut setelah bukti review konsisten: evaluasi koreksi label dan pemisahan rentang campuran sebagai dua eksperimen terpisah, menggunakan anotasi manual yang sudah ada. Notebook ini belum mengaktifkan aturan lirik di aplikasi dan tidak menjalankan training.
